sample data = gene + expression value
files for tgpt:
1. expression file.txt.gz
    1 line = 1 sample
    for each line - sorted gene names from highest to lowest
2. label file.txt.gz
    1 line = 1 label
    for each line - label name

for single cell data:
    adapter_premium/data/train.h5ad
    labels = cell types

for bulk data:
    data/0_data_for_mlp/TCGA-BRCA.star_tpm.csv'
    labels = survival time from other file

In [2]:
import numpy as np
import scipy.sparse
from tqdm import tqdm
import anndata as ad
import gzip
import shutil
import os
import pandas as pd
from transformers import PreTrainedTokenizerFast, GPT2LMHeadModel, GPT2Model


/scratch/2370352/conda/envs/myenv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Single Cell from Census x Gene

In [ ]:
# load data
adata_path = '../../adapter_premium/data_new/train.h5ad'
adata = ad.read_h5ad(adata_path)

In [7]:
# choose ony protein coding
with open("../../data/protein_coding_genes.txt", "r") as f:
    protein_coding_genes = [line.strip() for line in f]

protein_set = set(protein_coding_genes)

mask = adata.var["feature_name"].apply(lambda x: x in protein_set)
adata = adata[:, mask].copy()

NameError: name 'adata' is not defined

In [8]:
# check gene names in tGPT
tokenizer_file = "lixiangchun/transcriptome-gpt-1024-8-16-64" 
tokenizer = PreTrainedTokenizerFast.from_pretrained(tokenizer_file)
vocab = tokenizer.get_vocab()

In [27]:
# 1. Get the list of genes from your data (using the correct column)
# Based on your previous discovery, 'feature_name' contains the symbols
my_genes = set(adata.var['feature_name'].map(str).tolist()) 

# 2. Get the list of genes from the scGPT dictionary
# vocab.keys() contains the gene symbols the model was trained on
model_genes = set(vocab.keys())

# 3. Calculate the intersection (common genes) and missing genes
intersection = my_genes.intersection(model_genes)
missing = my_genes - model_genes

# 4. Display statistics
print(f"--- GENE COVERAGE STATISTICS ---")
print(f"Number of genes in your data:      {len(my_genes)}")
print(f"Number of genes in scGPT vocab:    {len(model_genes)}")
print(f"---------------------------------")
print(f"Common genes (intersection):       {len(intersection)}")
print(f"Missing genes in the model:        {len(missing)}")

coverage_pct = (len(intersection) / len(my_genes)) * 100
print(f"Data coverage percentage:          {coverage_pct:.2f}%")

# 5. Preview missing genes (optional)
if len(missing) > 0:
    print(f"\nSample missing genes: {list(missing)[:10]}")

--- GENE COVERAGE STATISTICS ---
Number of genes in your data:      19879
Number of genes in scGPT vocab:    21150
---------------------------------
Common genes (intersection):       18029
Missing genes in the model:        1850
Data coverage percentage:          90.69%

Sample missing genes: ['GFUS', 'H4C11', 'ENSG00000173862', 'LINC02692', 'ENSG00000269026', 'YAE1', 'ENSG00000268434', 'CFAP300', 'ENSG00000253896', 'ENSG00000256646']


In [ ]:
#keep only genes existing in tGPT vocab

mask = adata.var["feature_name"].apply(lambda x: x in model_genes)
adata = adata[:, mask].copy()

my_genes_new = set(adata.var['feature_name'].map(str).tolist()) 
len(my_genes_new)

In [46]:
def save_sc_data_for_ranking(adata, prefix, label_col='cell_type', top_n=2000):
    """
    Generuje zsynchronizowane pliki z rankingiem genów i etykietami.
    
    Args:
        adata: Obiekt AnnData
        prefix: Przedrostek nazw plików (np. 'my_data' stworzy 'my_data_rankings.txt.gz' itd.)
        label_col: Nazwa kolumny w adata.obs z etykietami
        top_n: Ile top genów zapisać dla każdej komórki
    """
    rankings_file = f"{prefix}_gene_rankings.txt.gz"
    labels_file = f"{prefix}_labels.txt.gz"
    
    gene_names = adata.var['feature_name'].values
    is_sparse = scipy.sparse.issparse(adata.X)
    
    print(f"Rozpoczynam generowanie plików dla {adata.n_obs} komórek...")
    
    # Otwieramy oba pliki jednocześnie w bloku 'with'
    with gzip.open(rankings_file, 'wt', encoding='utf-8') as f_rank, \
         gzip.open(labels_file, 'wt', encoding='utf-8') as f_lab:
        
        for i in tqdm(range(adata.n_obs)):
            # 1. Zapis etykiety (synchronizacja wiersz po wierszu)
            label = str(adata.obs[label_col].iloc[i])
            f_lab.write(f"{label}\n")
            
            # 2. Wyciąganie i sortowanie genów
            if is_sparse:
                # Wyciągamy tylko dane dla konkretnego wiersza (i)
                row = adata.X[i].toarray().flatten()
            else:
                row = adata.X[i]
            
            # Sprawdzenie czy komórka nie jest pusta
            if np.sum(row > 0) == 0:
                f_rank.write("\n")
                continue
                
            # Pobranie indeksów top N genów
            top_indices = np.argsort(row)[-top_n:][::-1]
            ranked_genes = gene_names[top_indices]
            
            # Zapis do pliku rankingów
            f_rank.write(" ".join(ranked_genes) + "\n")

    print(f"\nSukces! Utworzono pliki:")
    print(f" - {rankings_file}")
    print(f" - {labels_file}")



In [47]:
# --- PRZYKŁAD UŻYCIA ---
save_sc_data_for_ranking(adata, prefix="Muris_export", label_col="cell_type", top_n=2000)

Rozpoczynam generowanie plików dla 22400 komórek...


100%|██████████| 22400/22400 [01:04<00:00, 348.98it/s]


Sukces! Utworzono pliki:
 - Muris_export_gene_rankings.txt.gz
 - Muris_export_labels.txt.gz


In [ ]:
# check

with gzip.open('Muris_export_labels.txt.gz', 'rt') as f:  # 'rt' oznacza read text (tryb tekstowy)
    for _ in range(30):  # pokaż pierwsze 10 linii
        line = f.readline()
        if not line:
            break
        print(line.strip())

luminal epithelial cell of mammary gland
B cell
macrophage
basal-myoepithelial cell of mammary gland
basal-myoepithelial cell of mammary gland
T cell
luminal epithelial cell of mammary gland
macrophage
basal-myoepithelial cell of mammary gland
macrophage
macrophage
T cell
malignant cell
fibroblast of mammary gland
T cell
T cell
T cell
luminal epithelial cell of mammary gland
T cell
basal-myoepithelial cell of mammary gland
macrophage
macrophage
luminal epithelial cell of mammary gland
basal-myoepithelial cell of mammary gland
B cell
T cell
T cell
B cell
T cell
fibroblast of mammary gland


# Bulk from TCGA

In [3]:
data_path = '../../data/0_data_for_mlp/TCGA-OV.star_tpm.csv'
df = pd.read_csv(data_path)

In [4]:
df

,Unnamed: 0,TNMD,DPM1,SCYL3,C1orf112,FGR,CFH,FUCA2,GCLC,NFYA,...,RP1-20C7.7,RP11-57C19.8,C8orf44,C8orf44-SGK3,NPBWR1,RP11-18C24.9,CDR1,ACTL10,RP11-159G9.5,RP11-384O8.2
0,TCGA-04-1331,0.399445,6.964604,3.087055,2.830357,1.821261,4.551959,6.209091,2.395529,4.821098,...,0.000000,0.0,3.025525,0.110096,0.047678,0.0,0.0,0.0,0.0,1.777409
1,TCGA-04-1332,0.706553,6.357790,2.366196,2.358199,2.625551,4.963719,5.289868,2.338767,3.958128,...,0.000000,0.0,3.099497,0.000000,0.000000,0.0,0.0,0.0,0.0,1.406265
2,TCGA-04-1337,0.056167,7.952876,2.072312,1.397474,2.746227,3.988021,4.914899,3.064297,3.240314,...,0.000000,0.0,2.730792,0.069427,0.010064,0.0,0.0,0.0,0.0,1.901340
3,TCGA-04-1338,0.070389,6.422475,2.108324,1.537545,2.356904,4.106382,4.401173,2.922750,4.786476,...,0.000000,0.0,1.936289,0.136716,0.049770,0.0,0.0,0.0,0.0,0.200630
4,TCGA-04-1341,0.210888,6.403525,1.707348,1.402777,1.460323,2.099666,5.925715,2.307224,2.834731,...,0.000000,0.0,1.932439,0.000000,0.000000,0.0,0.0,0.0,0.0,1.772941
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
417,TCGA-61-2113,0.049631,7.205198,2.937702,3.163161,2.793022,5.917055,5.665970,3.248838,5.701677,...,0.000000,0.0,2.862689,0.073409,0.035061,0.0,0.0,0.0,0.0,1.846433
418,TCGA-OY-A56P,0.176323,7.019546,3.781958,3.202433,1.746270,4.516652,4.624411,3.688930,6.154494,...,0.000000,0.0,3.581869,0.255561,0.095452,0.0,0.0,0.0,0.0,1.663026
419,TCGA-OY-A56Q,0.087463,6.735166,3.440500,2.572308,1.534460,3.685021,4.916897,3.725905,6.236355,...,0.000000,0.0,3.413337,0.169027,0.015783,0.0,0.0,0.0,0.0,1.326997
420,TCGA-VG-A8LO,0.150040,6.748208,3.252506,2.756447,1.412456,5.040818,5.362309,3.669117,5.241813,...,0.060601,0.0,3.940204,0.235114,0.013784,0.0,0.0,0.0,0.0,1.913263


In [5]:
all_gene_columns = df.columns[1:]

In [9]:
model_genes = set(vocab.keys())
common_genes = [g for g in all_gene_columns if g in model_genes]
missing_count = len(all_gene_columns) - len(common_genes)

print(f"--- GENE FILTERING REPORT ---")
print(f"Genes in table:       {len(all_gene_columns)}")
print(f"Genes in model vocab: {len(model_genes)}")
print(f"Common genes:         {len(common_genes)}")
print(f"Missing (removed):    {missing_count}")
print(f"Final coverage:       {(len(common_genes)/len(all_gene_columns))*100:.2f}%")

--- GENE FILTERING REPORT ---
Genes in table:       20260
Genes in model vocab: 21150
Common genes:         18243
Missing (removed):    2017
Final coverage:       90.04%


In [10]:
df_filtered = df[['Unnamed: 0'] + common_genes].copy()

In [11]:
output_file = "bulk_gene_rankings_ov.txt.gz"
top_n = 2000

In [12]:
print(f"\nGenerating rankings for {len(df_filtered)} samples...")

with gzip.open(output_file, 'wt', encoding='utf-8') as f:
    # Iterujemy po wierszach (każdy wiersz to jedna próbka)
    for i in tqdm(range(len(df_filtered))):
        # Pobieramy wartości ekspresji dla danej próbki (bez kolumny z ID)
        row_values = df_filtered.iloc[i, 1:].values.astype(float)
        gene_names = np.array(common_genes)
        
        # Sortowanie indeksów od najwyższej ekspresji
        # argsort daje indeksy rosnąco, więc bierzemy końcówkę i odwracamy [::-1]
        top_indices = np.argsort(row_values)[-top_n:][::-1]
        
        # Pobieramy nazwy genów dla tych indeksów
        ranked_genes = gene_names[top_indices]
        
        # Zapisujemy linię rozdzieloną spacjami
        f.write(" ".join(ranked_genes) + "\n")

print(f"\nSuccess! Rankings saved to: {output_file}")


Generating rankings for 422 samples...


100%|██████████| 422/422 [00:02<00:00, 152.35it/s]


Success! Rankings saved to: bulk_gene_rankings_ov.txt.gz
